[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rome777/pose-image-tool/blob/main/pose_tool.ipynb)

# 원하는 포즈로 이미지 만들기 (ControlNet OpenPose 튜토리얼)

참조 사진 한 장의 **자세만** 가져와서, **다른 인물·다른 장면**을 같은 자세로 그리는 도구입니다.

프롬프트로는 자세를 정확히 지시하기 어렵습니다. "왼팔을 들고"라고 적어도 모델이 제멋대로 둡니다.
ControlNet은 참조 그림에서 **사람 관절 위치**만 뽑아 생성 조건으로 넣어 주는 보조 회로입니다.
그래서 **자세는 참조 사진을, 내용·조명·화풍은 프롬프트를** 따르게 됩니다.

## 전체 흐름

```mermaid
flowchart LR
    ref["참조 사진"] -- OpenPose --> skel["관절 뼈대 그림"]
    skel -- 자세 --> gen["SDXL + ControlNet"]
    prompt["프롬프트<br/>(나머지 다섯 칸)"] -- 내용·조명·화풍 --> gen
    gen --> out["결과 이미지"]
```

> Colab은 Mermaid를 그림으로 그려 주지 않아 위 블록이 코드로 보입니다.
> 그려진 그림은 [GitHub의 README](https://github.com/rome777/pose-image-tool#도구-설명)에서 보십시오.

## 실행 순서

1. **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 먼저 바꾸십시오. 안 바꾸면 2번 셀에서 멈춥니다.
2. 위에서부터 셀을 차례로 실행합니다. (`Ctrl+F9` 로 전체 실행해도 됩니다.)
3. 처음 실행하면 모델 내려받기에 3~5분쯤 걸립니다. 그 뒤로는 한 장에 20~30초입니다.
4. 결과는 `out/` 에 쌓이고, 마지막 셀이 `pose_tool_outputs.zip` 으로 묶어 내려받습니다.

## 재현에 대해

이 노트는 **참조 사진까지 코드로 만듭니다.** 남의 사진을 쓰지 않으므로 저작권 문제가 없고,
시드만 같으면 누가 언제 돌려도 같은 참조 사진이 나옵니다.
물론 손에 있는 사진을 쓸 수도 있습니다. `REFERENCE_SOURCE` 를 `"upload"` 로 바꾸십시오.

생성할 때마다 프롬프트·시드·모델 이름·스텝·가이던스를 `out/{이름}.json` 에 함께 적습니다.
**시드를 안 적어 두면 내일 같은 그림을 다시 못 뽑습니다.**

---
## 1. GPU 확인

무엇이 붙었는지부터 봅니다. 무료 티어에 배정되는 GPU는 그때그때 다르고, 특히 VRAM 용량이 중요합니다.
모델 파라미터가 통째로 여기 올라가야 계산이 되기 때문입니다. 이 노트는 T4(약 15GB)를 기준으로 맞췄습니다.

In [ ]:
# 이 셀이 하는 일: GPU가 붙었는지, 붙었다면 무엇이고 VRAM이 얼마인지 확인한다.
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU가 안 붙었습니다. 런타임 > 런타임 유형 변경 > T4 GPU 로 바꾸고 다시 실행하세요.")

p = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"{p.name} / VRAM {p.total_memory / 1024**3:.1f}GB / compute capability {major}.{minor}")

---
## 2. 설치

- `diffusers` : 확산 모델을 이름만 대면 내려받아 실행해 주는 라이브러리
- `controlnet_aux` : 사진에서 관절·윤곽선 같은 **조건 그림을 뽑아 주는** 전처리 모음 (OpenPose가 여기 들어 있음)

설치 후 런타임 재시작을 요구하면 재시작하고 이 셀부터 다시 실행하십시오.

In [ ]:
# 이 셀이 하는 일: 필요한 라이브러리를 설치한다. 2~3분 걸린다.
!pip install -q -U diffusers transformers accelerate safetensors
!pip install -q controlnet_aux
print("설치 완료")

---
## 3. 설정 — 재현에 필요한 값을 한자리에 모읍니다

여기 있는 값이 결과를 좌우합니다. 흩어 놓으면 나중에 무엇을 바꿨는지 알 수 없으니 한 칸에 모읍니다.

| 값 | 의미 | 왜 이렇게 두는가 |
|---|---|---|
| `BASE_MODEL` | 그림을 그리는 본체 | SDXL 1.0. 전신 인물이 비교적 안정적이고 포즈 ControlNet이 잘 갖춰져 있음 |
| `CONTROLNET_MODEL` | 관절 정보를 넣는 보조 회로 | SDXL용 OpenPose ControlNet |
| `VAE_MODEL` | 잠재 공간을 픽셀로 펴내는 장치 | SDXL을 float16으로 돌리면 결과가 통째로 검게 나오는 알려진 문제가 있어 수정판을 쓴다 |
| `STEPS` / `GUIDANCE` | 잡음을 걷어 내는 횟수 / 프롬프트를 미는 세기 | **증류되지 않은 기본 모델**이므로 스텝 25, 가이던스 5.0. (증류판이면 스텝 4, 가이던스 0.0) |
| `CONTROLNET_SCALE` | 자세를 얼마나 세게 강제할지 | 1.0에 가까울수록 자세는 정확해지지만 그림이 뻣뻣해진다. 0.8이 절충점 |
| `SEED` | 잡음의 출발점 | **고정해야 프롬프트를 고친 효과인지 운인지 구별할 수 있다** |

In [ ]:
# 이 셀이 하는 일: 모델 이름과 생성 값을 한자리에 모은다. 실험할 때 여기만 고친다.
BASE_MODEL       = "stabilityai/stable-diffusion-xl-base-1.0"
CONTROLNET_MODEL = "thibaud/controlnet-openpose-sdxl-1.0"
VAE_MODEL        = "madebyollin/sdxl-vae-fp16-fix"   # float16에서 검은 이미지가 나오는 문제를 막는다

STEPS            = 25      # 증류 모델이 아니므로 한 자릿수로 두면 안 된다
GUIDANCE         = 5.0     # 증류판이면 여기가 0.0 이어야 한다
CONTROLNET_SCALE = 0.8     # 자세 강제 세기 (0.0 = 무시, 1.0 = 최대)
WIDTH, HEIGHT    = 768, 1024
SEED             = 20260916

NEGATIVE = "cropped, out of frame, extra limbs, deformed hands, text, watermark"

import os, json, hashlib
os.makedirs("out", exist_ok=True)
print("설정 완료 ·", BASE_MODEL, "+", CONTROLNET_MODEL)

---
## 4. 파이프라인 한 번만 올리기

`diffusers` 가 돌려주는 실행 단위를 **파이프라인**이라고 부릅니다.
문장을 좌표로 바꾸고, 잡음에서 시작해 여러 번 걷어 내고, 마지막에 그림으로 펴내는 과정이 이 안에 묶여 있습니다.

**반드시 반복문 밖에서 한 번만 올립니다.** 반복문 안에서 올리면 이미지 한 장마다 모델을 다시 적재해서 백 배쯤 느려집니다.

여기서는 파이프라인을 두 개 만들되 **가중치는 공유합니다.**

- `t2i` : 참조 사진을 만들 때 쓰는 보통의 text-to-image
- `pose2i` : 관절 그림을 조건으로 받는 ControlNet 파이프라인

`t2i.components` 를 그대로 넘겨 주면 같은 UNet·VAE·텍스트 인코더를 재사용하므로 VRAM이 두 배로 들지 않습니다.

In [ ]:
# 이 셀이 하는 일: 모델을 GPU에 올린다. 처음 한 번만 3~5분쯤 내려받는다.
import torch
from diffusers import (StableDiffusionXLPipeline, StableDiffusionXLControlNetPipeline,
                       ControlNetModel, AutoencoderKL)

DTYPE = torch.float16

vae = AutoencoderKL.from_pretrained(VAE_MODEL, torch_dtype=DTYPE)

t2i = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, vae=vae, torch_dtype=DTYPE, variant="fp16", use_safetensors=True).to("cuda")

controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=DTYPE).to("cuda")

# 가중치를 공유해 두 번째 파이프라인을 만든다. 무게가 한 벌뿐이라 VRAM이 두 배로 들지 않는다.
pose2i = StableDiffusionXLControlNetPipeline(**t2i.components, controlnet=controlnet)

# VAE를 조각내어 디코딩하면 큰 이미지에서 메모리가 덜 든다
# (enable_model_cpu_offload() 는 쓰지 않는다. 두 파이프라인이 같은 모듈을 공유하므로
#  양쪽에 걸면 훅이 두 겹으로 붙어 매 스텝 CPU와 GPU를 오가느라 아주 느려진다.)
t2i.vae.enable_slicing()

print("파이프라인 준비 완료 · 최대 VRAM", f"{torch.cuda.max_memory_allocated()/1024**3:.1f}GB")

---
## 5. 참조 사진 준비

자세를 가져올 사진입니다. 세 가지 방법을 둡니다.

- `"generate"` (기본) — 같은 모델로 참조 사진까지 만듭니다. 저작권 걱정이 없고 시드만 같으면 재현됩니다.
- `"upload"` — 내 사진을 올립니다. Colab 파일 선택 창이 뜹니다.
- `"url"` — 인터넷 주소에서 받아 옵니다.

**좋은 참조 사진의 조건**은 분명합니다. 사람이 **한 명**, **전신이 다 들어오고**, **팔다리가 몸에 겹치지 않아야** 합니다.
관절이 가려지면 OpenPose가 못 찾고, 못 찾은 관절은 조건으로 들어가지 않습니다.

In [ ]:
# 이 셀이 하는 일: 자세를 가져올 참조 사진을 확보한다.
REFERENCE_SOURCE = "generate"   # "generate" | "upload" | "url"

REFERENCE_PROMPTS = {
    "pose_01": ("Full body studio photograph of one adult in a plain gray t-shirt and jeans, "
                "standing on a seamless light gray backdrop, feet apart, left arm raised straight "
                "up above the head, right arm extended horizontally to the side, facing the camera. "
                "Even soft studio lighting from the front, 50mm lens, the whole body from head to "
                "shoes inside the frame. Photorealistic reference photograph, neutral colors."),
    "pose_02": ("Full body studio photograph of one adult in a plain gray t-shirt and jeans, "
                "crouching on one knee on a seamless light gray backdrop, right knee down, left "
                "forearm resting on the left knee, head turned to the left, side three-quarter view. "
                "Even soft studio lighting from the front, 50mm lens, the whole body from head to "
                "shoes inside the frame. Photorealistic reference photograph, neutral colors."),
}

from PIL import Image
import torch

references = {}

if REFERENCE_SOURCE == "generate":
    for name, prompt in REFERENCE_PROMPTS.items():
        g = torch.Generator(device="cuda").manual_seed(SEED)
        img = t2i(prompt=prompt, negative_prompt=NEGATIVE, num_inference_steps=STEPS,
                  guidance_scale=GUIDANCE, width=WIDTH, height=HEIGHT, generator=g).images[0]
        img.save(f"out/{name}_ref.png")
        references[name] = img
        print(name, "참조 사진 생성 완료")

elif REFERENCE_SOURCE == "upload":
    from google.colab import files
    for fname, data in files.upload().items():
        open(fname, "wb").write(data)
        name = f"pose_{len(references)+1:02d}"
        img = Image.open(fname).convert("RGB").resize((WIDTH, HEIGHT))
        img.save(f"out/{name}_ref.png")
        references[name] = img

elif REFERENCE_SOURCE == "url":
    import requests, io
    URLS = {"pose_01": "https://example.com/your-photo.jpg"}
    for name, url in URLS.items():
        img = Image.open(io.BytesIO(requests.get(url, timeout=60).content)).convert("RGB")
        img = img.resize((WIDTH, HEIGHT))
        img.save(f"out/{name}_ref.png")
        references[name] = img

print("참조 사진", len(references), "장")

In [ ]:
# 이 셀이 하는 일: 참조 사진을 눈으로 확인한다.
from PIL import Image

def show(images, titles, scale=0.35):
    """여러 장을 가로로 붙여 한 번에 본다."""
    imgs = [im.resize((int(im.width*scale), int(im.height*scale))) for im in images]
    w, h = sum(i.width for i in imgs), max(i.height for i in imgs)
    canvas = Image.new("RGB", (w, h), "white")
    x = 0
    for im in imgs:
        canvas.paste(im, (x, 0)); x += im.width
    print(" | ".join(titles))
    return canvas

show(list(references.values()), list(references.keys()))

---
## 6. 포즈 추출 — 사진에서 관절만 뽑아냅니다

OpenPose는 사진에서 사람의 관절 위치를 찾아 **뼈대 그림**으로 그려 줍니다.
색깔 막대 하나가 뼈 하나, 점 하나가 관절 하나입니다. ControlNet은 원본 사진을 보지 않습니다.
**이 뼈대 그림만** 조건으로 받습니다. 그래서 인물·옷·배경은 프롬프트가 자유롭게 바꿀 수 있는 것입니다.

뼈대가 엉성하게 나오면 참조 사진을 바꾸는 편이 빠릅니다. 프롬프트로는 해결되지 않습니다.

In [ ]:
# 이 셀이 하는 일: 참조 사진에서 OpenPose로 관절 뼈대를 뽑는다.
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")

poses = {}
for name, img in references.items():
    skeleton = openpose(img, hand_and_face=False,
                        detect_resolution=512, image_resolution=max(WIDTH, HEIGHT))
    skeleton = skeleton.resize((WIDTH, HEIGHT))
    skeleton.save(f"out/{name}.png")       # 제출물의 pose_XX.png 가 이것이다
    poses[name] = skeleton
    print(name, "포즈 추출 완료")

show(list(poses.values()), list(poses.keys()))

---
## 7. 생성 함수 — 만들 때마다 기록을 함께 남깁니다

여기가 이 도구의 심장입니다. 두 가지를 지킵니다.

1. **시드를 문자열에서 만들되, 실행할 때마다 달라지지 않게 합니다.**
   파이썬 기본 `hash()` 는 보안상의 이유로 프로그램을 새로 실행할 때마다 결과가 달라집니다.
   그걸로 시드를 만들면 **오늘 뽑은 그림을 내일 다시 못 뽑습니다.** 그래서 `sha256` 을 씁니다.
2. **프롬프트·시드·모델·스텝·가이던스를 JSON으로 같이 저장합니다.**
   프롬프트만 저장하고 시드를 빼먹으면 재현이 안 됩니다.

In [ ]:
# 이 셀이 하는 일: 포즈 + 프롬프트로 한 장 만들고, 재현에 필요한 값을 JSON으로 남긴다.
import hashlib, json, torch

def stable_seed(key: str) -> int:
    """문자열 -> 항상 같은 정수. 파이썬 기본 hash()는 실행마다 달라지므로 쓰면 안 된다."""
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:8], 16)

def generate(out_name: str, pose_name: str, prompt: str, seed: int | None = None,
             controlnet_scale: float | None = None):
    scale = CONTROLNET_SCALE if controlnet_scale is None else controlnet_scale
    used_seed = SEED if seed is None else seed

    g = torch.Generator(device="cuda").manual_seed(used_seed)
    image = pose2i(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        image=poses[pose_name],                 # 조건으로 들어가는 것은 뼈대 그림이다
        controlnet_conditioning_scale=scale,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH, height=HEIGHT,
        generator=g,
    ).images[0]

    image.save(f"out/{out_name}.png")
    record = {
        "output": f"{out_name}.png",
        "pose_reference": f"{pose_name}.png",
        "prompt": prompt,
        "negative_prompt": NEGATIVE,
        "seed": used_seed,
        "base_model": BASE_MODEL,
        "controlnet_model": CONTROLNET_MODEL,
        "steps": STEPS,
        "guidance_scale": GUIDANCE,
        "controlnet_conditioning_scale": scale,
        "size": [WIDTH, HEIGHT],
    }
    with open(f"out/{out_name}.json", "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    print(f"{out_name}.png 저장 · seed={used_seed} · scale={scale}")
    return image

print("준비 완료 · stable_seed('test') =", stable_seed("test"))

---
## 8. 실험 1 — 자세는 그대로, 프롬프트만 바꿉니다

같은 뼈대에 서로 다른 인물·장소·화풍을 얹습니다.
프롬프트는 앞에서 배운 **여섯 칸**(피사체 / 동작 / 카메라 / 조명 / 환경 / 스타일)을 채워 씁니다.
단, **동작 칸은 참조 사진이 이미 정했으므로** 자세를 다시 묘사하지 않습니다.
자세를 글로 또 적으면 뼈대와 충돌합니다.

In [ ]:
# 이 셀이 하는 일: 같은 포즈(pose_01)에 프롬프트만 바꿔 세 장을 만든다.
EXP1 = {
    "output_01": ("A factory worker in a white safety helmet and navy work uniform inside a large "
                  "manufacturing plant. Medium full shot, static camera, 50mm lens. Soft morning "
                  "light from tall windows on the left throws long reflections on the polished "
                  "concrete floor. Rows of idle machinery recede into shallow depth of field. "
                  "Photorealistic documentary photography, muted industrial color palette."),
    "output_01b": ("An astronaut in a white spacesuit on the dusty surface of Mars. Full shot, "
                   "static camera, 35mm lens. Hard low sunlight from the right, long sharp shadow "
                   "on red sand. Distant rocky ridges under a pale orange sky. Photorealistic "
                   "cinematic still, warm desaturated palette."),
    "output_01c": ("A traditional Korean scholar in a pale blue hanbok standing in a temple "
                   "courtyard. Full shot, static camera, 85mm lens. Warm late afternoon light from "
                   "the left, soft shadows on stone tiles. Wooden pillars and a tiled roof behind, "
                   "softly out of focus. Painterly illustration, ink and watercolor texture."),
}

exp1_images = []
for name, prompt in EXP1.items():
    exp1_images.append(generate(name, "pose_01", prompt))

show([references["pose_01"], poses["pose_01"]] + exp1_images,
     ["참조", "뼈대"] + list(EXP1.keys()))

---
## 9. 실험 2 — 프롬프트는 그대로, 자세만 바꿉니다

이번에는 반대입니다. 같은 문장에 다른 뼈대를 넣습니다.
**인물·옷·장소는 그대로이고 자세만 바뀌면** 포즈 조건이 제대로 걸린 것입니다.

In [ ]:
# 이 셀이 하는 일: 같은 프롬프트에 다른 포즈(pose_02)를 넣어 비교한다.
SAME_PROMPT = EXP1["output_01"]   # 실험 1의 첫 프롬프트와 완전히 동일

img2 = generate("output_02", "pose_02", SAME_PROMPT)

show([references["pose_02"], poses["pose_02"], exp1_images[0], img2],
     ["참조(pose_02)", "뼈대(pose_02)", "output_01 (pose_01)", "output_02 (pose_02)"])

---
## 10. 자세를 얼마나 세게 강제할지 — `controlnet_conditioning_scale`

이 값 하나만 바꿔 봅니다. 낮으면 프롬프트가 이기고, 높으면 뼈대가 이깁니다.
"자세가 안 따라온다"는 문제는 프롬프트를 고칠 일이 아니라 **이 값을 올릴 일**입니다.

In [ ]:
# 이 셀이 하는 일: conditioning scale 만 바꿔 자세 반영 정도를 비교한다.
scale_images = []
for s in (0.3, 0.8, 1.2):
    scale_images.append(generate(f"scale_{str(s).replace('.', '')}", "pose_01",
                                 EXP1["output_01"], controlnet_scale=s))

show([poses["pose_01"]] + scale_images, ["뼈대", "scale 0.3", "scale 0.8", "scale 1.2"])

---
## 11. 재현성 확인

**"제출한 결과를 같은 실험으로 다시 만들 수 있는가"** 를 말이 아니라 숫자로 확인합니다.
같은 시드·같은 프롬프트·같은 뼈대로 한 번 더 뽑아, 앞의 결과와 픽셀이 완전히 같은지 비교합니다.

In [ ]:
# 이 셀이 하는 일: 같은 조건으로 다시 뽑아 픽셀이 동일한지 확인한다.
import numpy as np
from PIL import Image

again = generate("repro_check", "pose_01", EXP1["output_01"])

a = np.asarray(Image.open("out/output_01.png").convert("RGB"), dtype=np.int16)
b = np.asarray(again.convert("RGB"), dtype=np.int16)
diff = np.abs(a - b)

print("최대 픽셀 차이:", diff.max(), "/ 평균:", round(float(diff.mean()), 4))
print("완전히 동일한가:", bool(diff.max() == 0))

# 문자열에서 시드를 만드는 방식도 실행과 무관하게 같은 값이어야 한다
print("stable_seed('output_01') =", stable_seed("output_01"), "(몇 번을 새로 실행해도 같아야 함)")

---
## 12. 결과 내려받기

In [ ]:
# 이 셀이 하는 일: out/ 을 zip 으로 묶어 내려받는다.
import shutil
shutil.make_archive("pose_tool_outputs", "zip", "out")
print(sorted(os.listdir("out")))

from google.colab import files
files.download("pose_tool_outputs.zip")

---
## 13. 무엇을 바꿔 보았고, 어떻게 달라졌는가

> 아래는 위 셀을 무료 Colab T4에서 실제로 돌리고 적은 것입니다. 한 장에 약 26초(25스텝, 1.05s/it)였습니다.

### 먼저, 참조 사진부터가 증거였습니다

`pose_01` 참조 사진을 만들 때 프롬프트에 **"왼팔을 머리 위로 들고 오른팔을 옆으로 뻗은"** 이라고 적었습니다.
그런데 나온 사진은 **두 팔을 내리고 정면을 보고 선 평범한 자세**였습니다. 요청한 팔 동작이 통째로 무시된 것입니다.

이게 이 도구가 필요한 이유입니다. **자세는 프롬프트로 지시해도 잘 안 듣습니다.**
그래서 뼈대를 따로 넣어 주는 것입니다. 실습 첫 단계부터 그 한계가 그대로 드러났습니다.

### 프롬프트만 바꿨을 때 (실험 1)

`pose_01` 뼈대(정면·차렷에 가까운 서 있는 자세) 하나에 공장 노동자 / 우주 비행사 / 한복 차림 선비를 얹었습니다.

- **자세는 세 장 모두 같게 유지**됐습니다. 정면을 보고, 두 팔을 내리고, 발을 어깨너비로 벌린 그대로입니다.
- 바뀐 것은 인물·옷·배경·화풍뿐입니다. 세 번째 장은 지시한 대로 수채화 일러스트로 나왔습니다.
- 프롬프트에는 자세를 한 글자도 적지 않았습니다.

### 포즈만 바꿨을 때 (실험 2)

프롬프트를 한 글자도 안 바꾸고 뼈대만 `pose_02`(한쪽 무릎을 꿇고 앞쪽 무릎에 손을 얹은 자세)로 바꿨습니다.

- **인물과 장소는 그대로, 자세만 바뀌었습니다.** 안전모와 남색 작업복, 공장 배경이 유지된 채
  참조 사진의 무릎 꿇은 자세를 거의 그대로 따라갔습니다.
- 즉 **두 조건이 서로 다른 칸을 담당합니다.** 프롬프트는 "무엇이 있는가", 뼈대는 "어떤 자세인가"입니다.

### 자세 강제 세기 (`controlnet_conditioning_scale`)

| 값 | 실제로 나온 것 |
|---|---|
| 0.3 | **인물이 뒤로 돌아섰습니다.** 서 있다는 것만 맞고 방향과 팔 위치는 뼈대와 어긋났습니다 |
| 0.8 | 뼈대를 그대로 따르면서 화풍·조명 지시도 살아 있습니다. 여기가 절충점이었습니다 |
| 1.2 | 자세는 정확한데 **손이 무너지고**, 손 자리에 주인 없는 기계 부품 같은 것이 붙었습니다 |

"자세가 안 따라온다"는 문제는 프롬프트를 고칠 일이 아니라 **이 값을 올릴 일**입니다.
다만 올릴수록 손이 나빠지므로 무작정 1.0을 넘기지 않는 편이 낫습니다.

### 재현성

같은 시드·같은 프롬프트·같은 뼈대로 다시 뽑은 `repro_check.png` 는 `output_01.png` 와
**픽셀 차이 0**, 파일 해시까지 동일했습니다. `scale_08.png` 도 기본값과 같은 조건이라 같은 파일이 나왔습니다.

그래서 `out/{이름}.json` 에 시드까지 적어 두는 것이 중요합니다. 프롬프트만 적어 두면 재현이 안 됩니다.
문자열에서 시드를 만들 때 파이썬 기본 `hash()` 를 쓰면 실행할 때마다 값이 달라져 같은 문제가 생깁니다.

### 잘 안 되는 것

- **참조 사진에서 가려진 관절은 조건에 없습니다.** 팔이 몸통 뒤로 숨은 사진을 넣으면 그 팔은 모델이 제멋대로 그립니다.
  이때 프롬프트를 고치는 것은 소용이 없고 **참조 사진을 바꿔야** 합니다.
- **손가락**은 여전히 무너집니다. `hand_and_face=False` 로 돌려 손 관절이 조건에 없고,
  잠재 공간에서 손이 차지하는 자리가 원래 좁습니다. 손이 크게 잡히는 구도는 피하는 편이 낫습니다.
- **뼈대와 프롬프트가 충돌하면** 둘 다 아닌 그림이 나옵니다. 앉은 자세 뼈대에 "걸어가는 사람"이라고 적으면 망가집니다.
  프롬프트에서 **동작 칸은 비워 두는 것이 맞습니다.** 그 칸은 이미 뼈대가 채웠습니다.
- `enable_model_cpu_offload()` 를 **두 파이프라인에 모두 걸면** 안 됩니다. 같은 모듈을 공유하므로 훅이 두 겹으로 붙어
  매 스텝 CPU와 GPU를 오가느라 한 장에 몇 분씩 걸립니다. 처음 돌렸을 때 실제로 이 일이 일어났습니다.